<a href="https://colab.research.google.com/github/puderzuckerimschnee-max/bxmid03-hs26-team-2/blob/main/checks/Team2_U-Wert_Berechnung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# Team 2 · U-Wert der Aussenwände berechnen, ins Modell schreiben, prüfen (MVP v0.2)

Weitergeführt aus dem Starter-Notebook unseres Dozenten (`Team2_U-Wert_Starter.ipynb`).
Unser Soll-Prozess: **Bauphysik liefert Aufbau → U-Wert → Modell → IDS.**

| Schritt | Was passiert |
|---|---|
| 1 | Aussenwände finden (`Pset_WallCommon.IsExternal`) und verdächtige Wände markieren |
| 2 | Aufbautabelle `aufbau.csv` lesen (d, λ oder R, Quelle pro Schicht) |
| 3 | U = 1 / (Rsi + Σ d/λ + Rse) nach DIN EN ISO 6946, inkl. hinterlüfteter Fassaden |
| 4 | Handrechnung als Kontrolle |
| 5 | Ergebnis pro Wand gegen GEG Anlage 1 (0.28 W/(m²K)): bestanden / durchgefallen / fehlend |
| 6 | U-Wert als `Pset_WallCommon.ThermalTransmittance` in eine **Kopie** des IFC schreiben |
| 7 | IDS erzeugen und Original und Kopie prüfen |
| 8 | Negativfall: gleiche Regel, dünnere Dämmung |
| 9 | Bezug zu unserer Heizlast-Doku (DIN EN 12831): Transmissionswärmeverlust der Aussenwände |

Modell: `models/BIMcollab_Example_ARC.ifc` (IFC2X3, Längen in mm).
Werkzeuge: Python 3.12, IfcOpenShell 0.9.0.post1, ifctester 0.9.0, pandas.

## 1 · Pakete installieren (in Colab nötig, lokal nur einmal)

In [ ]:
%pip install -q ifcopenshell==0.9.0.post1 ifctester==0.9.0 pandas

## 2 · Modell und Aufbautabelle laden
Lokal aus dem Repository, sonst (Colab) von GitHub.

In [ ]:
import os, urllib.request
import pandas as pd

ROH = "https://raw.githubusercontent.com/puderzuckerimschnee-max/bxmid03-hs26-team-2/main/"

def hole(pfad_repo):
    """Datei aus dem Repository: lokal (../pfad) oder in Colab von GitHub laden."""
    lokal = os.path.join("..", pfad_repo)
    if os.path.exists(lokal):
        return lokal
    name = os.path.basename(pfad_repo)
    if not os.path.exists(name):
        urllib.request.urlretrieve(ROH + pfad_repo, name)
    return name

MODELL = hole("models/BIMcollab_Example_ARC.ifc")
AUFBAU_DATEI = "aufbau.csv" if os.path.exists("aufbau.csv") else hole("checks/aufbau.csv")
ERGEBNIS_ORDNER = "../results" if os.path.isdir("../results") else "."
IDS_ORDNER = "../requirements" if os.path.isdir("../requirements") else "."

import ifcopenshell, ifcopenshell.util.element as ue, ifcopenshell.util.unit as uu
modell = ifcopenshell.open(MODELL)
skala = uu.calculate_unit_scale(modell)   # Modelleinheit → m
print(f"{MODELL}: {modell.schema}, Länge × {skala} = m, {len(modell.by_type('IfcWall'))} Wände")

## 3 · Aussenwände
Nur Wände mit `IsExternal = TRUE`. Fehlende Werte bleiben leer, nicht 0.

In [ ]:
def schichten(w):
    mat = ue.get_material(w, should_skip_usage=True)
    if mat is None:
        return []
    if mat.is_a("IfcMaterialLayerSet"):
        return [l.Material.Name if l.Material else None for l in mat.MaterialLayers]
    if mat.is_a("IfcMaterialList"):
        return [m.Name for m in mat.Materials]
    return [getattr(mat, "Name", None)]

zeilen = []
for w in modell.by_type("IfcWall"):
    p = ue.get_psets(w)
    common = p.get("Pset_WallCommon", {})
    mengen = p.get("BaseQuantities", {}) or p.get("Qto_WallBaseQuantities", {})
    typ = ue.get_type(w)
    breite = mengen.get("Width")
    zeilen.append({
        "GlobalId": w.GlobalId,
        "Name": w.Name,
        "Wandtyp": typ.Name if typ else w.Name,
        "IsExternal": common.get("IsExternal"),
        "Dicke_Modell_m": round(breite * skala, 3) if breite else None,
        "Flaeche_netto_m2": mengen.get("NetSideArea"),   # Flächeneinheit im Modell: m²
        "Schichten_Modell": " | ".join(str(s) for s in schichten(w)),
        "U_im_Modell": common.get("ThermalTransmittance"),
    })
alle = pd.DataFrame(zeilen)
print(alle["IsExternal"].value_counts(dropna=False).to_string())
aussen = alle[alle["IsExternal"] == True].copy()
print(f"\n{len(aussen)} Aussenwände, davon mit U-Wert im Modell: {aussen['U_im_Modell'].notna().sum()}")

### Stimmt `IsExternal`? (Befund für den Architekten)
Eine Aussenwand mit «internal» im Typnamen oder unter 15 cm Dicke ist verdächtig. Diese Wände werden trotzdem geprüft, bekommen aber einen Hinweis.

In [ ]:
def verdacht(z):
    gruende = []
    if "internal" in str(z.Wandtyp).lower():
        gruende.append("Typname sagt 'internal'")
    if pd.notna(z.Dicke_Modell_m) and z.Dicke_Modell_m < 0.15:
        gruende.append(f"nur {z.Dicke_Modell_m*1000:.0f} mm dick")
    return "; ".join(gruende)

aussen["Hinweis_IsExternal"] = aussen.apply(verdacht, axis=1)
typen = (aussen.groupby(["Wandtyp", "Dicke_Modell_m", "Hinweis_IsExternal"], dropna=False)
         .size().rename("Anzahl").reset_index().sort_values("Anzahl", ascending=False))
print(typen.to_string(index=False))
print(f"\nVerdächtig: {(aussen['Hinweis_IsExternal'] != '').sum()} von {len(aussen)} Aussenwänden")

## 4 · Aufbautabelle (unsere Fachleistung)

`aufbau.csv`: eine Zeile pro Schicht, **von innen nach aussen**, pro Schicht entweder `d_m` und `lambda_W_mK` oder direkt `R_m2K_W`.

- `belueftet = stark`: hinterlüftete Fassade. Nach DIN EN ISO 6946:2018-03, Abschnitt 6.9.4, werden diese Luftschicht und alle Schichten weiter aussen **nicht** gerechnet, und aussen gilt Rse = Rsi = 0.13.
- Ruhende Luftschichten (Abschnitt 6.9.2): R aus Tabelle 8. Offene Stossfugen in der Aussenschale von zweischaligem Mauerwerk gelten laut Anmerkung zu 6.9.2 **nicht** als Lüftungsöffnungen.
- λ-Werte: DIN 4108-4:2020-11 Tabellen 1 und 2, ergänzend DIN EN ISO 10456:2010-05 Tabelle 3 (Stahlbeton, Sperrholz). Im Original geprüft am 03.11.2026.
- `Quelle`: woher der Wert kommt. **Keine Werte ohne Quelle.**
- `Status`: `Vorschlag` = noch nicht von uns geprüft, `geprüft` = von uns kontrolliert.

Die Schichtdicken hat das Modell nicht. Sie sind so angenommen, dass ihre Summe der Wanddicke im Modell entspricht.

In [ ]:
aufbau = pd.read_csv(AUFBAU_DATEI, sep=";", encoding="utf-8-sig")
print(f"{AUFBAU_DATEI}: {len(aufbau)} Schichten, {aufbau['Wandtyp'].nunique()} Wandtypen")
print("Status:", aufbau["Status"].value_counts().to_dict())
fehlende_typen = sorted(set(aussen["Wandtyp"]) - set(aufbau["Wandtyp"]))
print("Wandtypen ohne Aufbau:", fehlende_typen or "keine")
aufbau[["Wandtyp", "Nr", "Material_Annahme", "d_m", "lambda_W_mK", "R_m2K_W", "belueftet", "Quelle"]]

## 5 · U-Wert berechnen (DIN EN ISO 6946:2018-03)

- U = 1 / R_tot (Gleichung 1), R = d / λ (Gleichung 3), R_tot = Rsi + R1 + … + Rn + Rse (Gleichung 4)
- Wärmeübergangswiderstände bei horizontalem Wärmestrom (Tabelle 7): **Rsi = 0.13, Rse = 0.04 m²K/W**; bei stark belüfteter Luftschicht aussen Rse = Rsi = 0.13 (6.9.4)
- Endergebnis U auf **zwei signifikante Stellen** runden (6.5.2), z.B. 0.169 → 0.17. Der Vergleich mit dem Grenzwert nutzt den ungerundeten Wert.
- Vereinfachung: Holzständer in Dämmschichten (inhomogene Schicht nach 6.7.2) sind **nicht** berücksichtigt; der U-Wert der Timberframe-Wand ist dadurch etwas zu günstig.

In [ ]:
RSI, RSE = 0.13, 0.04

def u_wert(schichten_df):
    """U in W/(m²K) oder None, wenn eine gerechnete Schicht unvollständig ist."""
    summe_r, rse = 0.0, RSE
    for _, s in schichten_df.sort_values("Nr").iterrows():
        if str(s.get("belueftet", "")).strip().lower() == "stark":
            rse = RSI          # stark belüftet: ab hier nichts mehr rechnen, aussen wie innen
            break
        if pd.notna(s["R_m2K_W"]):
            summe_r += s["R_m2K_W"]
        elif pd.notna(s["d_m"]) and pd.notna(s["lambda_W_mK"]) and s["lambda_W_mK"] > 0:
            summe_r += s["d_m"] / s["lambda_W_mK"]
        else:
            return None
    return 1 / (RSI + summe_r + rse)

### Handrechnung als Kontrolle

**a) Test unseres Dozenten:** 0.20 m Dämmung mit λ = 0.035 → R = 5.714, U = 1 / (0.13 + 5.714 + 0.04) = **0.170**

**b) Eine Wand aus dem Modell: «Timberframe internal wall - limestone»** (von innen nach aussen)

| Schicht | d [m] | λ [W/(m·K)] | R = d/λ [m²K/W] |
|---|---|---|---|
| Rsi | | | 0.130 |
| Kalksandstein 1800 | 0.100 | 0.99 | 0.101 |
| EPS λD 0.035 | 0.060 | 0.036 | 1.667 |
| Folie | 0 | | 0.000 |
| Mineralwolle λD 0.035 | 0.140 | 0.036 | 3.889 |
| Hinterlüftung + Holzschalung | | | nicht gerechnet |
| Rse (= Rsi, hinterlüftet) | | | 0.130 |
| **Summe R_T** | | | **5.917** |

U = 1 / 5.917 = **0.169 W/(m²K)** ≤ 0.28 → bestanden

In [ ]:
test = pd.DataFrame([{"Nr": 1, "d_m": 0.20, "lambda_W_mK": 0.035, "R_m2K_W": None, "belueftet": None}])
assert abs(u_wert(test) - 0.170) < 0.001
print(f"a) Test Dozent bestätigt: U = {u_wert(test):.3f} W/(m²K)")

hand = 1 / (0.13 + 0.100/0.99 + 0.060/0.036 + 0 + 0.140/0.036 + 0.13)
code_u = u_wert(aufbau[aufbau["Wandtyp"] == "Timberframe internal wall - limestone"])
assert abs(hand - code_u) < 0.0005, (hand, code_u)
print(f"b) Handrechnung Timberframe: U = {hand:.3f}, Code: {code_u:.3f} → stimmt überein")

In [ ]:
u_typ = {}
for typ, gruppe in aufbau.groupby("Wandtyp"):
    u = u_wert(gruppe)
    u_typ[typ] = {"U_berechnet": None if u is None else round(u, 3),
                  "Dicke_Aufbau_m": round(gruppe["d_m"].sum(min_count=len(gruppe)), 3),
                  "Aufbau_Status": ", ".join(sorted(gruppe["Status"].dropna().unique()))}
u_typ = pd.DataFrame.from_dict(u_typ, orient="index")
u_typ.sort_values("U_berechnet")

## 6 · Ergebnis pro Wand

- **Grenzwert:** U ≤ 0.28 W/(m²K) für Aussenwände, Referenzgebäude nach **GEG Anlage 1, Zeile 1.1**.
  Achtung: Das ist ein *Referenzwert* für den Neubau-Nachweis, kein fester Höchstwert für jedes Bauteil. Wir nehmen ihn als Prüfwert.
- **Plausibilität:** Summe der Schichtdicken soll zur Wanddicke im Modell passen (± 5 mm).

In [ ]:
GRENZWERT = 0.28          # W/(m²K)
GRENZWERT_QUELLE = "GEG Anlage 1 Zeile 1.1 (Referenzgebäude, Aussenwand)"

def bewerten(z, grenzwert=GRENZWERT):
    hinweise = []
    if pd.notna(z.Dicke_Aufbau_m) and pd.notna(z.Dicke_Modell_m) and abs(z.Dicke_Aufbau_m - z.Dicke_Modell_m) > 0.005:
        hinweise.append(f"Aufbau {z.Dicke_Aufbau_m:.3f} m ≠ Modell {z.Dicke_Modell_m:.3f} m")
    if z.get("Hinweis_IsExternal"):
        hinweise.append("IsExternal prüfen: " + z.Hinweis_IsExternal)
    zusatz = ("; " + "; ".join(hinweise)) if hinweise else ""
    if pd.isna(z.U_berechnet):
        return "fehlend", "kein vollständiger Aufbau (d, λ oder R fehlt)" + zusatz
    if z.U_berechnet <= grenzwert:
        return "bestanden", f"U {z.U_berechnet:.3f} ≤ {grenzwert}" + zusatz
    return "durchgefallen", f"U {z.U_berechnet:.3f} > {grenzwert}" + zusatz

erg = aussen.join(u_typ, on="Wandtyp")
erg[["Ergebnis", "Grund"]] = erg.apply(lambda z: pd.Series(bewerten(z)), axis=1)
print(erg["Ergebnis"].value_counts().to_string())
print()
print(erg.groupby(["Wandtyp", "U_berechnet", "Ergebnis"]).size().rename("Anzahl").reset_index().to_string(index=False))

## 7 · U-Wert ins Modell schreiben (Kopie)
Nur Wände mit berechnetem U. Das Original bleibt unverändert.

In [ ]:
import ifcopenshell.api

def schreibe_kopie(ergebnis, ausgabe):
    kopie = ifcopenshell.open(MODELL)
    geschrieben = 0
    for _, z in ergebnis[ergebnis["U_berechnet"].notna()].iterrows():
        w = kopie.by_guid(z.GlobalId)
        pset = ue.get_pset(w, "Pset_WallCommon")
        pset = kopie.by_id(pset["id"]) if pset else ifcopenshell.api.run("pset.add_pset", kopie, product=w, name="Pset_WallCommon")
        ifcopenshell.api.run("pset.edit_pset", kopie, pset=pset, properties={"ThermalTransmittance": float(z.U_berechnet)})
        geschrieben += 1
    kopie.write(ausgabe)
    print(f"{geschrieben} Wände mit U-Wert → {ausgabe}")

AUSGABE_IFC = "BIMcollab_Example_ARC_mit_U.ifc"
schreibe_kopie(erg, AUSGABE_IFC)

## 8 · IDS-Prüfung
Regel: Jede Aussenwand (`IsExternal = true`) hat `Pset_WallCommon.ThermalTransmittance` ≤ 0.28.
Das IDS wird hier erzeugt und in `requirements/` gespeichert, damit es auch in IFC-Lite geladen werden kann.

In [ ]:
import ifctester.ids as ids, ifctester.reporter as reporter

def u_wert_ids():
    regel = ids.Ids(title="Team 2: U-Wert Aussenwände")
    spec = ids.Specification(name="Aussenwand U-Wert vorhanden und ≤ 0.28 (GEG Anlage 1)", ifcVersion=["IFC2X3", "IFC4"])
    spec.applicability.append(ids.Entity(name="IFCWALL"))
    spec.applicability.append(ids.Property(propertySet="Pset_WallCommon", baseName="IsExternal",
                                           value="true", dataType="IFCBOOLEAN"))
    spec.requirements.append(ids.Property(propertySet="Pset_WallCommon", baseName="ThermalTransmittance",
                                          dataType="IFCTHERMALTRANSMITTANCEMEASURE", cardinality="required",
                                          value=ids.Restriction(base="double", options={"maxInclusive": GRENZWERT})))
    regel.specifications.append(spec)
    return regel

IDS_DATEI = os.path.join(IDS_ORDNER, "Team2_U-Wert_Aussenwand.ids")
u_wert_ids().to_xml(IDS_DATEI)
print("IDS gespeichert:", IDS_DATEI)

def pruefe(datei):
    regel = u_wert_ids()             # pro Modell neu, sonst Fehler in ifctester
    m = ifcopenshell.open(datei)     # Referenz behalten, solange der Bericht läuft
    regel.validate(m)
    s = reporter.Json(regel).report()["specifications"][0]
    print(f"{os.path.basename(datei)}: {s['total_applicable_pass']} von {s['total_applicable']} Aussenwänden bestanden")
    return s["total_applicable_pass"], s["total_applicable"]

ids_ergebnis = {datei: pruefe(datei) for datei in (MODELL, AUSGABE_IFC)}

## 9 · Negativfall: gleiche Regel, dünnere Dämmung

Wir ändern **nur die Daten**: Beim Timberframe-Aufbau ist die Mineralwolle 40 mm statt 140 mm dick.
Die Regel (IDS, Grenzwert) bleibt gleich. Erwartung: Diese Wände fallen durch.

In [ ]:
aufbau_neg = aufbau.copy()
maske = (aufbau_neg["Wandtyp"] == "Timberframe internal wall - limestone") & (aufbau_neg["Schicht"] == "Insulation - plastic soft")
aufbau_neg.loc[maske, "d_m"] = 0.040

u_pos = u_wert(aufbau[aufbau["Wandtyp"] == "Timberframe internal wall - limestone"])
u_neg = u_wert(aufbau_neg[aufbau_neg["Wandtyp"] == "Timberframe internal wall - limestone"])
print(f"Timberframe positiv (140 mm MW): U = {u_pos:.3f} → {'bestanden' if u_pos <= GRENZWERT else 'durchgefallen'}")
print(f"Timberframe negativ ( 40 mm MW): U = {u_neg:.3f} → {'bestanden' if u_neg <= GRENZWERT else 'durchgefallen'}")
assert u_pos <= GRENZWERT < u_neg

erg_neg = erg.copy()
erg_neg.loc[erg_neg["Wandtyp"] == "Timberframe internal wall - limestone", "U_berechnet"] = round(u_neg, 3)
erg_neg[["Ergebnis", "Grund"]] = erg_neg.apply(lambda z: pd.Series(bewerten(z)), axis=1)
AUSGABE_IFC_NEG = "BIMcollab_Example_ARC_mit_U_negativ.ifc"
schreibe_kopie(erg_neg, AUSGABE_IFC_NEG)
ids_ergebnis[AUSGABE_IFC_NEG] = pruefe(AUSGABE_IFC_NEG)

## 10 · Bezug zu unserer Heizlast-Doku (DIN EN 12831)

In unserer Projektarbeit «Heizlastberechnung nach DIN EN 12831» (TH Lübeck, 2025) rechnen wir pro Bauteil
Φ_T = A × (U + ΔU_TB) × f_x × Δθ. Mit den berechneten U-Werten geht das jetzt pro Aussenwand aus dem Modell:

- ΔU_TB = 0.10 W/(m²K) (DIN EN 12831 NA, Tabelle NA.2, Fall D «alle anderen Fälle»)
- f_x = 1.0 (Aussenluft, Tabelle NA.26)
- Δθ = 20 °C − (−12 °C) = 32 K (wie in unserer Beispielrechnung)
- A = `NetSideArea` aus dem Modell (Netto-Wandfläche, Öffnungen abgezogen)

Hinweis: ΔU_TB gehört zur Heizlast, **nicht** zum U-Wert-Nachweis. Für die IDS-Prüfung bleibt U ohne Zuschlag.

In [ ]:
DELTA_U_TB, F_X, DELTA_THETA = 0.10, 1.0, 20 - (-12)
hl = erg[erg["U_berechnet"].notna() & erg["Flaeche_netto_m2"].notna()].copy()
hl["H_T_W_K"] = hl["Flaeche_netto_m2"] * (hl["U_berechnet"] + DELTA_U_TB) * F_X
hl["Phi_T_W"] = hl["H_T_W_K"] * DELTA_THETA
print(f"Aussenwände mit Fläche: {len(hl)} von {len(erg)}, Netto-Wandfläche {hl['Flaeche_netto_m2'].sum():.1f} m²")
print(f"Transmissionswärmeverlust Aussenwände: Φ_T = {hl['Phi_T_W'].sum()/1000:.2f} kW (bei Δθ = {DELTA_THETA} K)")
print(hl.groupby("Wandtyp")[["Flaeche_netto_m2", "Phi_T_W"]].sum().round(1).sort_values("Phi_T_W", ascending=False).to_string())

## 11 · Export nach `results/`

In [ ]:
def export(df, name):
    pfad = os.path.join(ERGEBNIS_ORDNER, name)
    (df.assign(Objekt=df["Name"].astype(str) + " (" + df["Wandtyp"].astype(str) + ")",
               Ist_U_W_m2K=df["U_berechnet"],
               Soll=f"≤ {GRENZWERT} W/(m²K) ({GRENZWERT_QUELLE})")
       [["GlobalId", "Objekt", "Dicke_Modell_m", "Ist_U_W_m2K", "Soll", "Ergebnis", "Grund"]]
       .assign(U_Angabe=lambda d: d["Ist_U_W_m2K"].map(lambda u: f"{u:#.2g}".rstrip(".") if pd.notna(u) else ""))
       .sort_values(["Ergebnis", "Objekt"])
       .to_csv(pfad, sep=";", index=False, encoding="utf-8-sig"))
    print("geschrieben:", pfad)

export(erg, "u_wert_aussenwaende.csv")
export(erg_neg, "u_wert_aussenwaende_negativfall.csv")

zusammenfassung = pd.DataFrame([
    {"Fall": "Original (ohne U-Werte)", "Datei": os.path.basename(MODELL)},
    {"Fall": "positiv: U berechnet aus aufbau.csv", "Datei": AUSGABE_IFC},
    {"Fall": "negativ: Timberframe mit 40 mm Mineralwolle", "Datei": AUSGABE_IFC_NEG},
])
zusammenfassung["IDS bestanden"] = [f"{ids_ergebnis[d][0]} von {ids_ergebnis[d][1]}" for d in (MODELL, AUSGABE_IFC, AUSGABE_IFC_NEG)]
print(zusammenfassung.to_string(index=False))

try:
    from google.colab import files
    # CSV, IDS und die beiden Modellkopien (je ca. 18 MB) für die Gegenprobe in IFC-Lite
    for f in ("u_wert_aussenwaende.csv", "u_wert_aussenwaende_negativfall.csv", "Team2_U-Wert_Aussenwand.ids",
              AUSGABE_IFC, AUSGABE_IFC_NEG):
        if os.path.exists(f):
            files.download(f)
except ImportError:
    pass

## 12 · Offen für uns (bitte prüfen, bevor wir Werte als «geprüft» markieren)

- Welches Material ist «Insulation - plastic hard» wirklich (EPS, XPS, PIR)? λ ändert das Ergebnis stark.
- Ist die Luftschicht im Kavitätsmauerwerk ruhend, schwach oder stark belüftet?
- Holzständer in den Dämmschichten (Timberframe, pl-iso-hsb): inhomogene Schicht nach DIN EN ISO 6946, 6.7.2, rechnen?
- 22 Wände «Limestone 100» und 8 «Timberframe *internal* wall» sind als Aussenwand markiert: mit dem Architekten klären.
- Grenzwert: GEG Anlage 1 ist ein Referenzwert. Passt er zu unserer Prüfaufgabe, oder nehmen wir GEG Anlage 7 (Höchstwerte bei Änderung)?